# Logistic Regression from Scratch — NumPy

We implement logistic regression with gradient descent using only NumPy,
then verify the result against scikit-learn.

**Update rule:**
$$\boldsymbol{\theta} := \boldsymbol{\theta} - \alpha \frac{1}{m} X^T (\sigma(X\boldsymbol{\theta}) - \mathbf{y})$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression as SklearnLR
from sklearn.metrics import accuracy_score
import sys
sys.path.insert(0, '../..')
from src.utils import set_style, sigmoid, plot_cost_history, classification_summary, plot_confusion_matrix

set_style()
np.random.seed(42)

## 1. Dataset

In [ ]:
X_raw, y = make_classification(
    n_samples=400,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    random_state=42,
)

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Add bias column
X_tr = np.hstack([np.ones((len(X_train), 1)), X_train])   # (m, 7)
X_te = np.hstack([np.ones((len(X_test), 1)),  X_test])

print(f'Train: {X_tr.shape}  Test: {X_te.shape}')

## 2. Logistic Regression Class

In [ ]:
class LogisticRegressionNP:
    """Binary logistic regression via gradient descent."""

    def __init__(self, alpha: float = 0.1, n_iter: int = 1000,
                 tol: float = 1e-6):
        self.alpha  = alpha
        self.n_iter = n_iter
        self.tol    = tol
        self.theta_       = None
        self.cost_history_ = []

    # ------------------------------------------------------------------
    def _cost(self, X, y, theta):
        m   = len(y)
        h   = sigmoid(X @ theta)
        h   = np.clip(h, 1e-15, 1 - 1e-15)
        return -float(y @ np.log(h) + (1 - y) @ np.log(1 - h)) / m

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'LogisticRegressionNP':
        m, n   = X.shape
        theta  = np.zeros(n)
        self.cost_history_ = []

        for i in range(self.n_iter):
            h        = sigmoid(X @ theta)
            gradient = X.T @ (h - y) / m
            theta    = theta - self.alpha * gradient

            cost = self._cost(X, y, theta)
            self.cost_history_.append(cost)

            # Early stopping
            if i > 0 and abs(self.cost_history_[-2] - cost) < self.tol:
                print(f'Converged at iteration {i}')
                break

        self.theta_ = theta
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return sigmoid(X @ self.theta_)

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        return (self.predict_proba(X) >= threshold).astype(int)

## 3. Train & Evaluate

In [ ]:
model = LogisticRegressionNP(alpha=0.5, n_iter=2000)
model.fit(X_tr, y_train)

plot_cost_history(model.cost_history_, title='Logistic Regression — Training Cost')

In [ ]:
y_pred_np = model.predict(X_te)

print('=== NumPy Logistic Regression ===')
classification_summary(y_test, y_pred_np, labels=['Class 0', 'Class 1'])
plot_confusion_matrix(y_test, y_pred_np, labels=['Class 0', 'Class 1'],
                      title='Confusion Matrix — NumPy Implementation')

## 4. Verify Against scikit-learn

In [ ]:
sk_model = SklearnLR(max_iter=2000, C=1e6)   # C very large → near-zero regularisation
sk_model.fit(X_train, y_train)
y_pred_sk = sk_model.predict(X_test)

print('=== scikit-learn LogisticRegression ===')
classification_summary(y_test, y_pred_sk, labels=['Class 0', 'Class 1'])

print(f'NumPy accuracy  : {accuracy_score(y_test, y_pred_np):.4f}')
print(f'sklearn accuracy: {accuracy_score(y_test, y_pred_sk):.4f}')

## 5. Probability Calibration Visualisation

In [ ]:
probs = model.predict_proba(X_te)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram of predicted probs coloured by truth
axes[0].hist(probs[y_test == 0], bins=20, color='#DC2626', alpha=0.7, label='True Class 0')
axes[0].hist(probs[y_test == 1], bins=20, color='#2563EB', alpha=0.7, label='True Class 1')
axes[0].axvline(0.5, color='black', lw=1.5, linestyle='--', label='Threshold=0.5')
axes[0].set_xlabel('Predicted P(y=1)')
axes[0].set_ylabel('Count')
axes[0].set_title('Predicted Probability Distribution')
axes[0].legend()

# Sorted predicted probabilities
sort_idx = np.argsort(probs)
axes[1].scatter(range(len(probs)), probs[sort_idx],
                c=y_test[sort_idx], cmap='coolwarm', alpha=0.7, s=20)
axes[1].axhline(0.5, color='black', lw=1.2, linestyle='--')
axes[1].set_xlabel('Sample (sorted by predicted prob)')
axes[1].set_ylabel('Predicted P(y=1)')
axes[1].set_title('Sorted Predictions (red=class0, blue=class1)')

plt.tight_layout()
plt.show()

## Summary

```python
# Core logistic regression gradient descent:
h        = sigmoid(X @ theta)          # predictions in (0, 1)
gradient = X.T @ (h - y) / m          # cross-entropy gradient
theta    = theta - alpha * gradient    # update
```

The result closely matches scikit-learn's implementation.

**Next:** [Apply to Kaggle Breast Cancer dataset →](03_kaggle_breast_cancer.ipynb)